# Auto etiquetado con modelos fundacionales multimodales

**Clase 6: práctica por grupos**

Vamos a construir el pipeline que usan por dentro herramientas como Roboflow Auto Label o Autodistill:

```
Prompt de texto ──► Grounding DINO ──► SAM 2 ──────► Dataset ──► YOLO
 "gloves . mask"      (cajas)          (máscaras)    (YOLO fmt)   (modelo chico)
```

**Dataset:** [CPPE-5](https://huggingface.co/datasets/rishitdagli/cppe-5), equipos de protección personal médica (5 clases). Es un dominio *distinto* a las imágenes web con las que se entrenaron los modelos, así que vamos a ver fallas reales.

**Truco:** CPPE-5 ya tiene etiquetas humanas. Vamos a **fingir que no las tenemos**, auto etiquetar las imágenes, y usar las etiquetas humanas solo para **medir** la calidad de las auto-etiquetas.

### Plan
| Parte | Qué hacemos | Quién |
|---|---|---|
| 0–2 | Cargar datos, probar Grounding DINO, medir la línea base | Todos |
| 3 | 🎯 **Desafío:** cada grupo mejora los prompts de **una clase** | Cada grupo |
| 4 | Máscaras con SAM 2 para su clase | Cada grupo |
| 5 | Juntamos los mejores prompts de todos los grupos y entrenamos YOLO | Todos / docente |

> Usar Google Colab con GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*. **Una sesión por grupo.**

In [ ]:
%pip install -q "transformers>=5.0" "supervision>=0.26" ultralytics datasets accelerate

In [ ]:
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv
import torch
from datasets import load_dataset
from supervision.metrics import MeanAveragePrecision
from tqdm.auto import tqdm
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()  # oculta avisos esperables (p. ej. "sam2_video → sam2")
warnings.filterwarnings("ignore", message=".*deterministic.*")  # avisos de PyTorch al entrenar YOLO

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
SEED = 42
N_TUNE = 50  # imágenes para experimentar (ajustar prompts/umbrales)
N_EVAL = 100  # imágenes para la evaluación final de cada grupo (¡no mirarlas mientras se ajusta!)
N_TEST = 100  # imágenes para evaluar el modelo YOLO final
N_POOL = N_TUNE + N_EVAL
BATCH_GD = 1  # Grounding DINO usa ~4.6 GB por imagen en el pico; con batch 2 ya se acerca al límite de una T4
WORKDIR = Path("tp5_work")
WORKDIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option("display.precision", 3)
print("device:", DEVICE)
if DEVICE == "cpu":
    print("⚠️ Sin GPU: va a funcionar, pero lento. Revisen el tipo de entorno en Colab.")

## 0. Datos

- **pool** = `TUNE` + `EVAL`: imágenes que vamos a auto etiquetar.
  - `TUNE` → para **experimentar** con prompts y umbrales.
  - `EVAL` → para la **evaluación final**. Si ajustamos y evaluamos sobre las mismas imágenes, el resultado es optimista.
- **test**: para evaluar el modelo YOLO del final.

CPPE-5 tiene algunos problemas de calidad (cajas fuera de la imagen, imágenes cuyo tamaño no coincide con las anotaciones): recortamos las cajas y descartamos esas imágenes. *Los datasets "humanos" tampoco son perfectos.*

In [ ]:
CLASES = ["Coverall", "Face_Shield", "Gloves", "Goggles", "Mask"]
MAX_LADO = 1333  # lado máximo de las imágenes (ahorra memoria)

ds = load_dataset("rishitdagli/cppe-5", split="train")
assert ds.features["objects"]["category"].feature.names == CLASES


def preparar(ejemplo):
    img = ejemplo["image"].convert("RGB")
    if img.size != (ejemplo["width"], ejemplo["height"]):
        return None  # anotaciones inconsistentes con la imagen → la descartamos
    xywh = np.array(ejemplo["objects"]["bbox"], dtype=np.float32).reshape(-1, 4)
    xyxy = np.concatenate([xywh[:, :2], xywh[:, :2] + xywh[:, 2:]], axis=1)
    escala = MAX_LADO / max(img.size)
    if escala < 1:  # hay imágenes de hasta 7329x6240; Grounding DINO igual las reduce a ≤1333
        img = img.resize((round(img.width * escala), round(img.height * escala)))
        xyxy = xyxy * escala
    w, h = img.size
    xyxy = np.clip(xyxy, 0, [w, h, w, h])
    cls = np.array(ejemplo["objects"]["category"], dtype=int)
    ok = (xyxy[:, 2] - xyxy[:, 0] > 1) & (xyxy[:, 3] - xyxy[:, 1] > 1)
    return img, sv.Detections(xyxy=xyxy[ok], class_id=cls[ok])


datos, descartadas = [], 0
for ejemplo in ds.shuffle(seed=SEED):
    r = preparar(ejemplo)
    if r is None:
        descartadas += 1
        continue
    datos.append(r)
    if len(datos) == N_POOL + N_TEST:
        break

pool_imgs = [img for img, _ in datos[:N_POOL]]
pool_gt = [gt for _, gt in datos[:N_POOL]]
test_imgs = [img for img, _ in datos[N_POOL:]]
test_gt = [gt for _, gt in datos[N_POOL:]]
TUNE = slice(0, N_TUNE)
EVAL = slice(N_TUNE, N_POOL)
print(f"pool: {len(pool_imgs)} (tune {N_TUNE} + eval {N_EVAL}) | test: {len(test_imgs)} | descartadas: {descartadas}")

conteo = pd.Series(np.concatenate([g.class_id for g in pool_gt])).map(dict(enumerate(CLASES))).value_counts()
conteo.to_frame("instancias en el pool")

In [ ]:
mask_annotator = sv.MaskAnnotator(opacity=0.5)


def dibujar(img, det, etiquetas=None, mascaras=False, colores=None):
    grosor = sv.calculate_optimal_line_thickness(img.size)
    kw = {} if colores is None else {"color": sv.ColorPalette.from_hex(colores)}
    box_annotator = sv.BoxAnnotator(thickness=grosor, **kw)
    label_annotator = sv.LabelAnnotator(
        text_scale=sv.calculate_optimal_text_scale(img.size), text_thickness=grosor, text_padding=3, **kw
    )
    if etiquetas is None:
        etiquetas = [
            CLASES[c] + ("" if det.confidence is None else f" {s:.2f}")
            for c, s in zip(det.class_id, det.confidence if det.confidence is not None else [None] * len(det))
        ]
    out = img.copy()
    if mascaras and det.mask is not None:
        out = mask_annotator.annotate(out, det)
    out = box_annotator.annotate(out, det)
    return label_annotator.annotate(out, det, labels=etiquetas)


def mostrar(imagenes, titulos=None, cols=4, alto=4.5):
    filas = int(np.ceil(len(imagenes) / cols))
    fig, axes = plt.subplots(filas, cols, figsize=(cols * alto, filas * alto))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for i, (ax, img) in enumerate(zip(axes, imagenes)):
        ax.imshow(img)
        if titulos:
            ax.set_title(titulos[i])
    plt.tight_layout()
    plt.show()


EJEMPLOS = [0, 1, 2, 3]
mostrar([dibujar(pool_imgs[i], pool_gt[i]) for i in EJEMPLOS], [f"GT humano #{i}" for i in EJEMPLOS])

## 1. Detección zero-shot con Grounding DINO

Grounding DINO recibe **una imagen y un texto** y devuelve cajas asociadas a frases del texto:
- El prompt es una lista de frases en minúscula separadas por `" . "`, por ejemplo `"gloves . face mask ."`.
- `box_threshold`: confianza mínima de la caja. `text_threshold`: confianza mínima para asignar una frase a la caja.
- El modelo devuelve la **frase** que activó la caja (p. ej. `"face mask"`), no un índice de clase. Necesitamos una **ontología**: un mapeo *prompt → clase del dataset* (la misma idea que usa Autodistill).

In [ ]:
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

GD_ID = "IDEA-Research/grounding-dino-tiny"
gd_processor = AutoProcessor.from_pretrained(GD_ID)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(DEVICE).eval()

# prompt (lo que "lee" Grounding DINO) → clase del dataset
ONTOLOGIA_BASE = {
    "coverall": "Coverall",
    "face shield": "Face_Shield",
    "gloves": "Gloves",
    "goggles": "Goggles",
    "mask": "Mask",
}


def frase_a_clase(frase, ontologia):
    # Grounding DINO devuelve la(s) palabra(s) del prompt que activaron la caja, p. ej. "face mask".
    # Buscamos a qué prompt de la ontología corresponde (si hay varios, el más largo).
    if not frase:
        return -1
    candidatos = [p for p in ontologia if p in frase or frase in p]
    if not candidatos:
        return -1
    destino = ontologia[max(candidatos, key=len)]
    return -1 if destino is None else CLASES.index(destino)  # None = frase "trampa": la caja se descarta


@torch.no_grad()
def detectar_gd(imagenes, ontologia, box_threshold=0.3, text_threshold=0.25, batch=BATCH_GD, progreso=False):
    texto = " . ".join(ontologia) + " ."
    resultados = []
    for i in tqdm(range(0, len(imagenes), batch), disable=not progreso):
        lote = imagenes[i : i + batch]
        inputs = gd_processor(images=lote, text=[texto] * len(lote), return_tensors="pt").to(DEVICE)
        outputs = gd_model(**inputs)
        salidas = gd_processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=box_threshold,
            text_threshold=text_threshold,
            target_sizes=[img.size[::-1] for img in lote],
        )
        for r in salidas:
            n = len(r["boxes"])
            frases = list(r["text_labels"])[:n]
            det = sv.Detections(
                xyxy=r["boxes"].cpu().numpy().reshape(-1, 4),
                confidence=r["scores"].cpu().numpy(),
                class_id=np.array([frase_a_clase(f, ontologia) for f in frases], dtype=int),
                data={"frase": np.array(frases, dtype=object)},
            )
            resultados.append(det[det.class_id >= 0])
    if DEVICE == "mps":
        torch.mps.empty_cache()  # en Mac la memoria de GPU es la RAM del sistema
    return resultados


preds = detectar_gd([pool_imgs[i] for i in EJEMPLOS], ONTOLOGIA_BASE, box_threshold=0.3)
mostrar(
    [dibujar(pool_imgs[i], pool_gt[i]) for i in EJEMPLOS] + [dibujar(pool_imgs[i], p) for i, p in zip(EJEMPLOS, preds)],
    [f"GT humano #{i}" for i in EJEMPLOS] + [f"Grounding DINO #{i}" for i in EJEMPLOS],
)

**Para discutir:** ¿qué clases detecta bien? ¿Cuáles confunde? ¿Hay cajas duplicadas o que agrupan varios objetos?

## 2. Línea base: ¿qué tan buenas son las auto-etiquetas?

"Se ve bien" no es una métrica. Auto etiquetamos el pool una sola vez con un umbral bajo (0.15). Después podemos probar distintos umbrales **sin volver a correr el modelo**: filtrar por confianza es gratis.

- **AP50 por clase**: calidad del ranking de cajas (IoU ≥ 0.5), independiente del umbral.
- **Precisión / recall a un umbral**: lo que efectivamente terminaría en el dataset.

In [ ]:
UMBRAL_MIN = 0.15
HISTORIAL = []  # intentos del desafío (sección 3); se acumulan entre corridas
BASE_TUNE = {}  # línea base por clase para el desafío (se calcula una vez)


def filtrar(dets, umbral):
    return [d[d.confidence >= umbral] for d in dets]


IOU_MIN = 0.5  # IoU mínimo para considerar que una caja "acierta" (Tarea D)


def emparejar(pred, gt, iou_min=None):
    # Matching greedy por clase (de mayor a menor confianza).
    # Devuelve: acierto por predicción, encontrado por GT.
    iou_min = IOU_MIN if iou_min is None else iou_min
    acierto = np.zeros(len(pred), dtype=bool)
    encontrado = np.zeros(len(gt), dtype=bool)
    for c in np.unique(np.concatenate([pred.class_id, gt.class_id])):
        ip, ig = np.where(pred.class_id == c)[0], np.where(gt.class_id == c)[0]
        if not (len(ip) and len(ig)):
            continue
        iou = sv.box_iou_batch(pred.xyxy[ip], gt.xyxy[ig])
        for k in np.argsort(-pred.confidence[ip]):
            cand = np.where(encontrado[ig], -1.0, iou[k])
            j = int(np.argmax(cand))
            if cand[j] >= iou_min:
                encontrado[ig[j]] = True
                acierto[ip[k]] = True
    return acierto, encontrado


def evaluar(preds, gts, umbral=0.3):
    r = MeanAveragePrecision().update(preds, gts).compute()
    ap50 = dict(zip(r.matched_classes, r.ap_per_class[:, 0]))
    tp, fp, fn = (np.zeros(len(CLASES)) for _ in range(3))
    for p, g in zip(filtrar(preds, umbral), gts):
        acierto, encontrado = emparejar(p, g)
        np.add.at(tp, p.class_id[acierto], 1)
        np.add.at(fp, p.class_id[~acierto], 1)
        np.add.at(fn, g.class_id[~encontrado], 1)
    tabla = pd.DataFrame(
        {
            "AP50": [ap50.get(c, np.nan) for c in range(len(CLASES))],
            "precisión": tp / np.maximum(tp + fp, 1),
            "recall": tp / np.maximum(tp + fn, 1),
            "GT": (tp + fn).astype(int),
        },
        index=CLASES,
    )
    tabla.loc["TOTAL (mAP50)"] = [r.map50, tp.sum() / max(tp.sum() + fp.sum(), 1), tp.sum() / max(tp.sum() + fn.sum(), 1), int((tp + fn).sum())]
    return tabla


t0 = time.time()
auto_base = detectar_gd(pool_imgs, ONTOLOGIA_BASE, box_threshold=UMBRAL_MIN, progreso=True)
print(f"{len(pool_imgs)} imágenes en {time.time() - t0:.0f}s")

linea_base = evaluar(auto_base[TUNE], pool_gt[TUNE], umbral=0.3)
linea_base

### Sensibilidad al umbral

Un cambio chico en el umbral mueve mucho la precisión y el recall, y el mejor umbral **no es el mismo para todas las clases**.

In [ ]:
filas = []
for u in np.round(np.arange(0.15, 0.61, 0.05), 2):
    t = evaluar(auto_base[TUNE], pool_gt[TUNE], umbral=u)
    for clase in CLASES:
        filas.append({"umbral": u, "clase": clase, "precisión": t.loc[clase, "precisión"], "recall": t.loc[clase, "recall"]})
barrido = pd.DataFrame(filas)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for clase, g in barrido.groupby("clase"):
    axes[0].plot(g["umbral"], g["precisión"], marker="o", label=clase)
    axes[1].plot(g["umbral"], g["recall"], marker="o", label=clase)
axes[0].set_title("Precisión vs box_threshold")
axes[1].set_title("Recall vs box_threshold")
for ax in axes:
    ax.set_xlabel("box_threshold")
    ax.grid(alpha=0.3)
axes[1].legend()
plt.show()

## 3. 🎯 Desafío por grupos

Cada grupo recibe **una clase** y tiene que conseguir las mejores auto-etiquetas para esa clase **cambiando solo el texto del prompt y el umbral**.

**Reglas**
1. Experimentar solo sobre `TUNE` (celda **3.1**, todas las veces que quieran).
2. Cuando estén conformes, **una sola** evaluación sobre `EVAL` (celda **3.2**).
3. Gana el grupo que más mejore el **AP50** respecto de la línea base de su clase.

⚠️ **Face_Shield** y **Goggles** tienen pocos ejemplos en `TUNE` (~15 objetos): un par de cajas mueven mucho el AP. Si la mejora es chica, puede ser ruido.

**Ideas de prompts:** sinónimos (*visor*, *eyewear*), atributos (*blue surgical mask*, *white protective suit*), singular vs plural (*glove* vs *gloves*), términos del dominio médico (*PPE gown*, *N95 respirator*), descripciones más largas. Recuerden el ejemplo de SAM: *"wiper"* vs *"wipers"*.

In [ ]:
# ============ CONFIGURACIÓN DEL GRUPO ============
GRUPO = "Grupo 1"
CLASE = "Goggles"  # una de: Coverall, Face_Shield, Gloves, Goggles, Mask
PROMPTS = ["goggles", "safety goggles", "protective glasses"]  # prueben 2–4 por vez
UMBRAL = 0.3  # umbral de confianza para precisión / recall
# =================================================

HISTORIAL = []  # se acumula con cada corrida de la celda 3.1

### 3.1 Experimentar (sobre `TUNE`)

Cada prompt se evalúa **solo** (ontología de una clase). La tabla acumula todos sus intentos.

La línea base es el prompt ingenuo de su clase, también **solo**. En la sección 2 ese mismo prompt tenía otro AP50 porque competía con las otras 4 frases: por ejemplo, las máscaras se las "queda" `mask` en vez de `goggles`.

In [ ]:
c = CLASES.index(CLASE)
gts_tune = [g[g.class_id == c] for g in pool_gt[TUNE]]
preds_tune = {}
for prompt in PROMPTS:
    preds_tune[prompt] = detectar_gd(pool_imgs[TUNE], {prompt: CLASE}, box_threshold=UMBRAL_MIN)
    fila = evaluar(preds_tune[prompt], gts_tune, umbral=UMBRAL).loc[CLASE]
    HISTORIAL.append({"clase": CLASE, "prompt": prompt, "umbral": UMBRAL, "AP50": fila["AP50"], "precisión": fila["precisión"], "recall": fila["recall"]})

prompt_base = list(ONTOLOGIA_BASE)[c]
if CLASE not in BASE_TUNE:  # línea base: el prompt ingenuo de la clase, solo, sobre TUNE
    BASE_TUNE[CLASE] = evaluar(detectar_gd(pool_imgs[TUNE], {prompt_base: CLASE}, box_threshold=UMBRAL_MIN), gts_tune).loc[CLASE, "AP50"]
print(f"Línea base '{CLASE}' (prompt '{prompt_base}' solo): AP50 = {BASE_TUNE[CLASE]:.3f} | objetos de {CLASE} en TUNE: {sum(len(g) for g in gts_tune)}")
historial = pd.DataFrame(HISTORIAL).query("clase == @CLASE").drop(columns="clase")
historial.drop_duplicates(["prompt", "umbral"], keep="last").sort_values("AP50", ascending=False)

### ¿Dónde se equivoca?

🟩 correcta · 🟥 falso positivo · 🟨 objeto que **no** detectó. Se muestran las imágenes de `TUNE` con más errores para el mejor prompt de la última corrida.

In [ ]:
def ver_errores(imgs, preds, gts, umbral, n=4):
    casos = []
    for img, p, g in zip(imgs, filtrar(preds, umbral), gts):
        acierto, encontrado = emparejar(p, g)
        faltan = g[~encontrado]
        det = sv.Detections.merge([
            sv.Detections(xyxy=p.xyxy.reshape(-1, 4), confidence=p.confidence, class_id=np.where(acierto, 0, 1)),
            sv.Detections(xyxy=faltan.xyxy.reshape(-1, 4), confidence=np.zeros(len(faltan)), class_id=np.full(len(faltan), 2)),
        ])
        etiquetas = [["OK", "FP", "no detectado"][k] + (f" {s:.2f}" if k < 2 else "") for k, s in zip(det.class_id, det.confidence)]
        errores = (~acierto).sum() + (~encontrado).sum()
        casos.append((errores, dibujar(img, det, etiquetas, colores=["#2ecc71", "#e74c3c", "#f1c40f"])))
    casos.sort(key=lambda x: -x[0])
    mostrar([im for _, im in casos[:n]], [f"{e} errores" for e, _ in casos[:n]], cols=2, alto=6)


mejor = max(PROMPTS, key=lambda p: evaluar(preds_tune[p], gts_tune, umbral=UMBRAL).loc[CLASE, "AP50"])
print(f"Mejor prompt de esta corrida: '{mejor}'")
ver_errores(pool_imgs[TUNE], preds_tune[mejor], gts_tune, UMBRAL)

**Para discutir en el grupo:** ¿los errores son falsos positivos o objetos no detectados? ¿Hay un patrón (objetos chicos, oclusión, confusión con otra clase)? ¿El umbral ayuda o hay que cambiar el prompt?

### 3.2 Evaluación final (sobre `EVAL`) — **una sola vez**

In [ ]:
# ============ ELECCIÓN FINAL DEL GRUPO ============
intentos = [h for h in HISTORIAL if h["clase"] == CLASE and not np.isnan(h["AP50"])]
PROMPT_FINAL = max(intentos, key=lambda h: h["AP50"])["prompt"]  # por defecto, el mejor AP50 en TUNE (pueden escribir otro)
UMBRAL_FINAL = UMBRAL
# ==================================================

gts_eval = [g[g.class_id == c] for g in pool_gt[EVAL]]
base = evaluar(detectar_gd(pool_imgs[EVAL], {list(ONTOLOGIA_BASE)[c]: CLASE}, box_threshold=UMBRAL_MIN), gts_eval, umbral=0.3).loc[CLASE]
final = evaluar(detectar_gd(pool_imgs[EVAL], {PROMPT_FINAL: CLASE}, box_threshold=UMBRAL_MIN), gts_eval, umbral=UMBRAL_FINAL).loc[CLASE]

print(
    f"{GRUPO} | {CLASE} | '{PROMPT_FINAL}' @ {UMBRAL_FINAL} | "
    f"AP50 {final['AP50']:.3f} (base {base['AP50']:.3f}, {final['AP50'] - base['AP50']:+.3f}) | "
    f"P {final['precisión']:.2f} | R {final['recall']:.2f}"
)

**Para discutir:** ¿la mejora que vieron en `TUNE` se sostuvo en `EVAL`? Si no, ¿por qué?

## 4. De cajas a máscaras con SAM 2

Usamos cada caja de su prompt final como **prompt** para SAM 2 (Clase 6). Así se obtiene un dataset de **segmentación** sin dibujar un solo polígono. (CPPE-5 no tiene máscaras humanas: acá solo inspeccionamos visualmente.)

In [ ]:
from transformers import Sam2Model, Sam2Processor

SAM_ID = "facebook/sam2.1-hiera-tiny"
sam_processor = Sam2Processor.from_pretrained(SAM_ID)
sam_model = Sam2Model.from_pretrained(SAM_ID).to(DEVICE).eval()  # el warning "sam2_video → sam2" es esperable


@torch.no_grad()
def segmentar(imagen, det):
    det = det[:]
    if len(det) == 0:
        return det
    inputs = sam_processor(images=imagen, input_boxes=[det.xyxy.tolist()], return_tensors="pt").to(DEVICE)
    outputs = sam_model(**inputs, multimask_output=False)
    masks = sam_processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"])[0]
    det.mask = masks[:, 0].numpy().astype(bool)
    return det


imgs_clase = [i for i in range(N_TUNE) if (pool_gt[i].class_id == c).any()][:4]
dets = filtrar(detectar_gd([pool_imgs[i] for i in imgs_clase], {PROMPT_FINAL: CLASE}, box_threshold=UMBRAL_MIN), UMBRAL_FINAL)
mostrar(
    [dibujar(pool_imgs[i], segmentar(pool_imgs[i], d), mascaras=True) for i, d in zip(imgs_clase, dets)],
    [f"'{PROMPT_FINAL}' + SAM 2" for _ in imgs_clase],
)

## Tareas

Para seguir explorando después del desafío. Elijan las que quieran, en cualquier orden.

| Tarea | Pregunta |
|---|---|
| **A. Frases trampa** | ¿Agregar frases que "absorban" falsos positivos mejora la precisión? |
| **B. Ambigüedad en SAM 2** | ¿Qué máscaras alternativas propone SAM 2 para una misma caja o punto? |
| **C. Vocabulario abierto** | ¿Qué detecta Grounding DINO fuera de este dataset? ¿Dónde falla? |
| **D. Métrica estricta vs laxa** | ¿Cuántos "errores" son en realidad cajas un poco corridas? |

Extra: busquen en las imágenes de errores casos donde la **etiqueta humana** esté mal o falte.

### Tarea A: Frases trampa

En la sección 2 vimos que un prompt rinde distinto cuando compite con otras frases: si el prompt incluye `mask`, las máscaras ya no se confunden con `goggles`. Acá lo usamos a propósito: agregamos frases que mapean a `None`. Las cajas que caigan en esas frases se **descartan**.

Prueben distintas trampas según los falsos positivos que vieron en *¿Dónde se equivoca?*.

In [ ]:
# ============ TAREA A ============
TRAMPAS = ["face", "person", "hand"]  # frases que "absorben" falsos positivos (se descartan)
# =================================

ontologia_trampa = {PROMPT_FINAL: CLASE, **{t: None for t in TRAMPAS if t != PROMPT_FINAL}}
filas = {}
for nombre, onto in [("sin trampas", {PROMPT_FINAL: CLASE}), (f"con trampas {TRAMPAS}", ontologia_trampa)]:
    preds = detectar_gd(pool_imgs[TUNE], onto, box_threshold=UMBRAL_MIN)
    filas[nombre] = evaluar(preds, gts_tune, umbral=UMBRAL_FINAL).loc[CLASE, ["AP50", "precisión", "recall"]]
print(f"'{PROMPT_FINAL}' sobre TUNE:")
pd.DataFrame(filas).T

### Tarea B: Ambigüedad en SAM 2

SAM es *ambiguity-aware* (Clase 6): para un mismo prompt propone **3 máscaras** con un puntaje de calidad (IoU estimado) cada una. Con una caja, las opciones suelen parecerse; con un **punto**, la ambigüedad es mayor (¿el punto marca las antiparras, la cara o la persona?).

In [ ]:
# ============ TAREA B ============
USAR_PUNTO = False  # False: prompt = caja de Grounding DINO | True: prompt = punto en el centro de la caja
IMAGEN = 0  # cuál de las imágenes de la sección 4 usar (0 a 3)
# =================================

i, d = [(i, d) for i, d in zip(imgs_clase, dets) if len(d)][IMAGEN]
caja = d[int(np.argmax(d.confidence))]
x1, y1, x2, y2 = caja.xyxy[0].tolist()
if USAR_PUNTO:
    prompt_sam = {"input_points": [[[[(x1 + x2) / 2, (y1 + y2) / 2]]]], "input_labels": [[[1]]]}
else:
    prompt_sam = {"input_boxes": [[[x1, y1, x2, y2]]]}

with torch.no_grad():
    inputs = sam_processor(images=pool_imgs[i], return_tensors="pt", **prompt_sam).to(DEVICE)
    outputs = sam_model(**inputs, multimask_output=True)
masks = sam_processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"])[0][0]  # (3, H, W)
scores = outputs.iou_scores[0, 0].cpu().numpy()

vistas = []
for m, s in zip(masks, scores):
    det = sv.Detections(xyxy=caja.xyxy, mask=m[None].numpy().astype(bool), class_id=caja.class_id)
    vistas.append(dibujar(pool_imgs[i], det, [f"IoU estimado {s:.2f}"], mascaras=True))
mostrar(vistas, [f"Máscara {k + 1} ({'punto' if USAR_PUNTO else 'caja'})" for k in range(3)], cols=3, alto=5)

### Tarea C: Vocabulario abierto

Grounding DINO no está limitado a las 5 clases de CPPE-5: prueben cualquier imagen (URL o una foto subida al panel de archivos de Colab) y cualquier texto.

Ideas: objetos comunes (*"laptop"*, *"backpack"*), atributos (*"red cup"*), relaciones (*"the cat on the left"*), **conteo** (*"two cats"*) y **negación** (*"a person without glasses"*). ¿Cuáles entiende y cuáles no? ¿Por qué un VLM (Clase 5) podría responder mejor?

In [ ]:
# ============ TAREA C ============
IMAGEN_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
RUTA_LOCAL = ""  # si suben una foto a Colab, pongan acá su ruta (p. ej. "/content/foto.jpg")
TEXTOS = ["cat", "remote control", "the cat on the left", "a cat without a remote"]
UMBRAL_LIBRE = 0.3
# =================================

import requests
from PIL import Image

img_libre = Image.open(RUTA_LOCAL if RUTA_LOCAL else requests.get(IMAGEN_URL, stream=True).raw).convert("RGB")

vistas = []
for texto in TEXTOS:
    with torch.no_grad():
        inputs = gd_processor(images=img_libre, text=texto.lower().strip(" .") + " .", return_tensors="pt").to(DEVICE)
        r = gd_processor.post_process_grounded_object_detection(
            gd_model(**inputs), inputs.input_ids, threshold=UMBRAL_LIBRE, text_threshold=0.25, target_sizes=[img_libre.size[::-1]]
        )[0]
    det = sv.Detections(xyxy=r["boxes"].cpu().numpy().reshape(-1, 4), confidence=r["scores"].cpu().numpy(),
                        class_id=np.zeros(len(r["scores"]), dtype=int))
    vistas.append(dibujar(img_libre, det, [f"{t} {s:.2f}" for t, s in zip(r["text_labels"], det.confidence)]))
mostrar(vistas, [f'"{t}"' for t in TEXTOS], cols=2, alto=5)

### Tarea D: Métrica estricta vs laxa

En *¿Dónde se equivoca?* aparecen casos donde una caja cubre antiparras + máscara: la caja "encontró" el objeto, pero el IoU con la etiqueta humana es < 0.5, así que cuenta **dos veces**: como falso positivo y como objeto no detectado. ¿Cuánto cambian precisión y recall si somos más laxos?

(El AP50 no cambia: por definición usa IoU 0.5.)

In [ ]:
# ============ TAREA D ============
IOUS = [0.3, 0.5, 0.7]
# =================================

preds = detectar_gd(pool_imgs[TUNE], {PROMPT_FINAL: CLASE}, box_threshold=UMBRAL_MIN)
filas = {}
for iou in IOUS:
    IOU_MIN = iou
    filas[f"IoU ≥ {iou}"] = evaluar(preds, gts_tune, umbral=UMBRAL_FINAL).loc[CLASE, ["precisión", "recall"]]
IOU_MIN = 0.5  # volvemos al valor estándar
print(f"'{PROMPT_FINAL}' @ {UMBRAL_FINAL} sobre TUNE:")
pd.DataFrame(filas).T

## 5. Todos juntos: ontología de la clase y destilación en YOLO

Completamos la ontología con el **prompt y umbral ganador de cada grupo**, auto etiquetamos el pool completo y entrenamos un detector **chico y rápido** (YOLO11n, ~2.6M parámetros):
- **A)** YOLO entrenado con las **etiquetas humanas** del pool (referencia).
- **B)** YOLO entrenado con las **auto-etiquetas** del pool (sin etiquetar nada a mano).

Ambos se evalúan en `test`: imágenes nunca vistas, con etiquetas humanas.

In [ ]:
# ======= COMPLETAR CON LOS RESULTADOS DE LOS GRUPOS =======
ONTOLOGIA_CLASE = {  # prompt ganador → clase
    "coverall": "Coverall",
    "face shield": "Face_Shield",
    "gloves": "Gloves",
    "goggles": "Goggles",
    "mask": "Mask",
}
UMBRALES_CLASE = {"Coverall": 0.3, "Face_Shield": 0.3, "Gloves": 0.3, "Goggles": 0.3, "Mask": 0.3}
# ===========================================================

if ONTOLOGIA_CLASE == ONTOLOGIA_BASE:
    auto_clase = auto_base  # todavía sin completar: reutilizamos la línea base
else:
    auto_clase = detectar_gd(pool_imgs, ONTOLOGIA_CLASE, box_threshold=UMBRAL_MIN, progreso=True)
umbral_por_caja = lambda d: np.array([UMBRALES_CLASE[CLASES[k]] for k in d.class_id])
auto_clase_filtrado = [d[d.confidence >= umbral_por_caja(d)] for d in auto_clase]

pd.concat(
    {
        "AP50 base": evaluar(auto_base[EVAL], pool_gt[EVAL])["AP50"],
        "AP50 clase": evaluar(auto_clase[EVAL], pool_gt[EVAL])["AP50"],
    },
    axis=1,
)

Si en el prompt conjunto alguna clase rinde peor que en el desafío, es porque ahora **todas las frases compiten entre sí** dentro del mismo prompt.

In [ ]:
import os

os.environ["YOLO_VERBOSE"] = "False"  # menos logs de entrenamiento
from ultralytics import YOLO

EPOCHS = 30  # con menos épocas ambos modelos quedan sub-entrenados y la comparación es ruido
YOLO_DEVICE = 0 if DEVICE == "cuda" else DEVICE


def exportar_yolo(nombre, imgs_train, dets_train):
    raiz = WORKDIR / f"yolo_{nombre}"
    for split, imgs, dets in [("train", imgs_train, dets_train), ("val", test_imgs, test_gt)]:
        (raiz / "images" / split).mkdir(parents=True, exist_ok=True)
        (raiz / "labels" / split).mkdir(parents=True, exist_ok=True)
        for i, (img, det) in enumerate(zip(imgs, dets)):
            img.save(raiz / "images" / split / f"{i:04d}.jpg")
            w, h = img.size
            lineas = [
                f"{k} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}"
                for (x1, y1, x2, y2), k in zip(det.xyxy, det.class_id)
            ]
            (raiz / "labels" / split / f"{i:04d}.txt").write_text("\n".join(lineas))
    yaml = raiz / "data.yaml"
    yaml.write_text(f"path: {raiz.resolve()}\ntrain: images/train\nval: images/val\nnames: {dict(enumerate(CLASES))}\n")
    return yaml


def entrenar(nombre, yaml):
    modelo = YOLO("yolo11n.pt")
    modelo.train(
        data=str(yaml), epochs=EPOCHS, imgsz=640, batch=16, seed=SEED, device=YOLO_DEVICE, workers=2,
        project=str((WORKDIR / "runs").resolve()), name=nombre, exist_ok=True, plots=False, verbose=False,
    )
    m = modelo.val(data=str(yaml), split="val", device=YOLO_DEVICE, plots=False, verbose=False)
    ap50 = dict(zip(m.box.ap_class_index, m.box.ap50))
    return pd.Series({**{CLASES[k]: ap50.get(k, 0.0) for k in range(len(CLASES))}, "mAP50": m.box.map50})


t0 = time.time()
res_gt = entrenar("gt", exportar_yolo("gt", pool_imgs, pool_gt))
res_auto = entrenar("auto", exportar_yolo("auto", pool_imgs, auto_clase_filtrado))
print(f"Entrenamiento: {(time.time() - t0) / 60:.1f} min")

resultados = pd.DataFrame({"A) etiquetas humanas": res_gt, "B) auto-etiquetas": res_auto})
resultados

**Para discutir:**
- ¿Qué porcentaje del rendimiento "humano" se recupera **sin etiquetar nada a mano**?
- ¿Las clases donde YOLO-auto rinde peor coinciden con las que tenían peores auto-etiquetas? Los errores del modelo base **se heredan** (homogeneización, Clase 7).
- Con 150 imágenes de entrenamiento y 100 de test, ¿qué diferencias son confiables y cuáles pueden ser ruido?